In [ ]:
"""
=============================================================================
BCSD Daily Pipeline v3 — Xu & Wang (2019)
Adapted for: ACCESS-CM2 Tmax → CHIRTS 0.05° over India
=============================================================================

FIXES over v2:
  ✓ Calendar mismatch: ACCESS-CM2 uses 'noleap' (365-day), CHIRTS uses
    Gregorian. Alignment on (year, month, day) not raw DOY.
  ✓ Leap years: Every Feb-29 in CHIRTS is explicitly tracked. GCM data on
    noleap calendar has no DOY 366, so QM pools are built correctly.
  ✓ NaN safety: Every NaN-producing operation is guarded. No silent ignoring.
  ✓ Grid alignment: Verified before any computation. Fails loudly if grids
    don't match.
  ✓ Shapefile masking: Applied after regridding but before QM so ocean cells
    are skipped (saves ~50% compute).
  ✓ Resume/checkpoint: Each step saves output; if file exists, loads it
    instead of recomputing.
  ✓ Yearly file loading: Handles separate per-year NetCDFs (glob pattern).
  ✓ 9-year running average edge handling: Explicit documentation + edge
    padding.

Data layout expected (from your Kaggle screenshot):
  /kaggle/input/YOUR_DATASET/chirtstmax_005_clean/  → 32 yearly .nc files
  /kaggle/input/YOUR_DATASET/chirtstmax_1deg/        → 32 yearly .nc files
  /kaggle/input/YOUR_DATASET/gcm_tasmax_regrid_1deg/ → 35 yearly .nc files
  /kaggle/input/YOUR_DATASET/target_shapefile/       → .shp + sidecar files

Paper answer: Xu & Wang 2019 used 13 INDIVIDUAL GCMs, each downscaled
  independently via BCSD, then compared as an ensemble. Your single-GCM
  (ACCESS-CM2) application is the same method applied once.
=============================================================================
"""

import numpy as np
import xarray as xr
import glob
import os
import json
import time as timer
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    # --- Data paths (EDIT THESE) ---
    CHIRTS_005_DIR = "/kaggle/input/datasets/kalpitasaha/bcsd-24-6-26/chirtstmax_005_clean"

    CHIRTS_1DEG_DIR = "/kaggle/input/datasets/kalpitasaha/bcsd-24-6-26/chirtstmax_1deg"

    CM_1DEG_DIR = "/kaggle/input/datasets/kalpitasaha/bcsd-24-6-26/gcm_tasmax_regrid_1deg"

    SHAPEFILE_DIR = "/kaggle/input/datasets/kalpitasaha/bcsd-24-6-26/target_shapefile"
    WORK_DIR = '/kaggle/working/bcsd_checkpoints'

    # --- Variable names in your NetCDFs ---
    OBS_VAR = 'Tmax'       # Variable name in CHIRTS files
    GCM_VAR = 'tasmax'      # Variable name in ACCESS-CM2 files

    # --- Periods ---
    TRAIN_START = 1983     # CHIRTS starts 1983
    TRAIN_END = 2014
    GCM_HIS_START = 1980   # GCM historical starts 1980
    GCM_HIS_END = 2014
    # Future: set these when you have future GCM data
    FUTURE_START = 2015
    FUTURE_END = 2099

    # --- BCSD parameters ---
    RUNNING_AVG_WINDOW = 9   # years (must be odd)
    QM_DAY_WINDOW = 15       # ±days for QM pooling

    # --- IDW parameters ---
    IDW_POWER = 2
    IDW_N_NEIGHBORS = 4

    # --- GCM calendar ---
            # not 'Tmax'
       # correct
         # not 1980 (CHIRTS starts 1983)
    GCM_CALENDAR = 'proleptic_gregorian'  # has leap years'  # ACCESS-CM2 uses noleap (365-day)
    # Set to 'standard' if your GCM has leap years


# =============================================================================
# STEP 0: DATA LOADING — handles yearly files, verifies grids
# =============================================================================

def load_yearly_files(directory, var_name, year_start, year_end,
                      file_pattern='*.nc'):
    """
    Load yearly NetCDF files from a directory into a single DataArray.

    Handles:
    • Automatic detection of file naming patterns
    • Sorting by year
    • Standardizing dim names to (time, lat, lon)
    • Ascending lat/lon sort

    Returns
    -------
    xr.DataArray (time, lat, lon)
    """
    files = sorted(glob.glob(os.path.join(directory, file_pattern)))
    if not files:
        raise FileNotFoundError(
            f"No files matching '{file_pattern}' in {directory}")

    print(f"    Found {len(files)} files in {os.path.basename(directory)}")

    datasets = []
    for f in files:
        ds = xr.open_dataset(f)

        # Find the variable
        if var_name in ds:
            da = ds[var_name]
        else:
            # Try case-insensitive match
            for v in ds.data_vars:
                if v.lower() == var_name.lower():
                    da = ds[v]
                    break
            else:
                raise KeyError(
                    f"Variable '{var_name}' not found in {f}. "
                    f"Available: {list(ds.data_vars)}")

        # Standardize dimension names
        rename = {}
        for d in list(da.dims) + list(da.coords):
            dl = d.lower()
            if dl in ('longitude', 'x') and d != 'lon':
                rename[d] = 'lon'
            if dl in ('latitude', 'y') and d != 'lat':
                rename[d] = 'lat'
        if rename:
            da = da.rename(rename)

        datasets.append(da)

    # Concatenate along time
    combined = xr.concat(datasets, dim='time')

    # Sort lat/lon ascending
    combined = combined.sortby('lat').sortby('lon')

    # Subset to requested years
    combined = combined.sel(
        time=slice(f'{year_start}-01-01', f'{year_end}-12-31'))

    # Convert to float32 for memory
    combined = combined.astype(np.float32)

    return combined


def verify_grid_alignment(da_a, da_b, name_a, name_b, tol=1e-4):
    """
    STRICT grid alignment check. Fails LOUDLY if grids don't match.

    Checks:
    • Same number of lat/lon points
    • Same coordinate values (within tolerance)
    • Reports exact mismatch details
    """
    print(f"  Verifying grid: {name_a} vs {name_b}...")

    # Shape check
    if da_a.sizes['lat'] != da_b.sizes['lat']:
        raise ValueError(
            f"LAT COUNT MISMATCH: {name_a} has {da_a.sizes['lat']} lats, "
            f"{name_b} has {da_b.sizes['lat']} lats")
    if da_a.sizes['lon'] != da_b.sizes['lon']:
        raise ValueError(
            f"LON COUNT MISMATCH: {name_a} has {da_a.sizes['lon']} lons, "
            f"{name_b} has {da_b.sizes['lon']} lons")

    # Coordinate value check
    lat_diff = np.abs(da_a.lat.values - da_b.lat.values)
    lon_diff = np.abs(da_a.lon.values - da_b.lon.values)

    if lat_diff.max() > tol:
        # Find first mismatch
        idx = np.argmax(lat_diff > tol)
        raise ValueError(
            f"LAT VALUES MISMATCH at index {idx}: "
            f"{name_a}={da_a.lat.values[idx]:.6f}, "
            f"{name_b}={da_b.lat.values[idx]:.6f}, "
            f"diff={lat_diff[idx]:.6f}")
    if lon_diff.max() > tol:
        idx = np.argmax(lon_diff > tol)
        raise ValueError(
            f"LON VALUES MISMATCH at index {idx}: "
            f"{name_a}={da_a.lon.values[idx]:.6f}, "
            f"{name_b}={da_b.lon.values[idx]:.6f}, "
            f"diff={lon_diff[idx]:.6f}")

    print(f"    ✓ Grids aligned: {da_a.sizes['lat']}×{da_a.sizes['lon']}, "
          f"max lat diff={lat_diff.max():.2e}, max lon diff={lon_diff.max():.2e}")


def check_nan_report(da, name):
    """Report NaN statistics for a DataArray. Fail if ALL NaN."""
    total = da.size
    nan_count = int(np.isnan(da.values).sum())
    nan_pct = 100 * nan_count / total

    # Per-cell: how many cells are fully NaN across all time?
    all_nan_cells = np.all(np.isnan(da.values), axis=0).sum()
    total_cells = da.sizes['lat'] * da.sizes['lon']

    print(f"    {name}: shape={da.shape}, "
          f"NaN={nan_count}/{total} ({nan_pct:.1f}%), "
          f"all-NaN cells={all_nan_cells}/{total_cells}")

    if nan_count == total:
        raise ValueError(f"FATAL: {name} is 100% NaN!")

    return {'total': total, 'nan_count': nan_count, 'nan_pct': nan_pct,
            'all_nan_cells': int(all_nan_cells)}


def apply_shapefile_mask(da, shp_dir):
    """
    Mask data to the study region defined by a shapefile.
    Returns da with ocean/outside cells set to NaN.
    """
    import geopandas as gpd
    import regionmask

    # Find .shp file in the directory
    shp_files = glob.glob(os.path.join(shp_dir, '*.shp'))
    if not shp_files:
        raise FileNotFoundError(f"No .shp file found in {shp_dir}")
    shp_path = shp_files[0]
    print(f"    Using shapefile: {os.path.basename(shp_path)}")

    gdf = gpd.read_file(shp_path)
    region = regionmask.from_geopandas(gdf)
    mask = region.mask(da.lon, da.lat)

    # mask is NaN outside the region, integer inside
    da_masked = da.where(~np.isnan(mask))
    n_inside = int((~np.isnan(mask)).sum())
    n_total = mask.size
    print(f"    Mask: {n_inside}/{n_total} cells inside region")

    return da_masked, mask


# =============================================================================
# CALENDAR ALIGNMENT — handles noleap vs Gregorian
# =============================================================================

def align_calendars(da_obs, da_gcm, gcm_calendar='noleap'):
    """
    Align observation (Gregorian) and GCM (possibly noleap) calendars.

    Problem: ACCESS-CM2 uses a 365-day 'noleap' calendar — Feb 29 never
    exists. CHIRTS uses Gregorian — Feb 29 exists in leap years. If we
    pair by raw timestamp, Feb 29 obs have no GCM match, and Mar 1 onward
    shifts by 1 day in leap years.

    Solution: Create a common day-index that both datasets share.
    For the QM step we don't need exact day-pairing (we pool ±15 days
    anyway), but we DO need consistent day-of-year numbering.

    Returns
    -------
    da_obs with extra coordinate 'common_doy' (1-365, Feb29 gets doy=60)
    da_gcm with extra coordinate 'common_doy' (1-365)
    leap_days_mask: boolean mask of Feb-29 days in obs
    """
    # For obs: assign common_doy (1-365 where Feb29 maps to doy=60)
    obs_times = da_obs.time.values
    obs_months = da_obs.time.dt.month.values
    obs_days = da_obs.time.dt.day.values
    obs_years = da_obs.time.dt.year.values

    # Identify leap day positions
    is_leap_day = (obs_months == 2) & (obs_days == 29)
    n_leap = is_leap_day.sum()
    print(f"    Obs has {n_leap} leap days (Feb 29)")

    # Compute common DOY: for non-leap years, use standard DOY
    # For leap years: Jan1-Feb28 → 1-59, Feb29 → 60, Mar1-Dec31 → 60-365
    # This maps every day to 1-365 (366 days in leap years get squeezed)
    common_doy_obs = np.zeros(len(obs_times), dtype=int)
    for i, (y, m, d) in enumerate(zip(obs_years, obs_months, obs_days)):
        if m == 2 and d == 29:
            common_doy_obs[i] = 60  # Feb 29 → same slot as Mar 1 in noleap
        else:
            # Day of year in a non-leap year
            import datetime
            try:
                # Use a non-leap year (2001) for consistent DOY
                ref = datetime.date(2001, m, d)
                common_doy_obs[i] = ref.timetuple().tm_yday
            except ValueError:
                # Shouldn't happen if we handled Feb29 above
                common_doy_obs[i] = 0

    da_obs = da_obs.assign_coords(common_doy=('time', common_doy_obs))

    # For GCM: straightforward DOY 1-365 (no leap days)
    if gcm_calendar == 'noleap':
        gcm_doy = da_gcm.time.dt.dayofyear.values
    else:
        # If GCM has leap years too, apply same logic
        gcm_months = da_gcm.time.dt.month.values
        gcm_days = da_gcm.time.dt.day.values
        gcm_years = da_gcm.time.dt.year.values
        gcm_doy = np.zeros(len(da_gcm.time), dtype=int)
        for i, (y, m, d) in enumerate(zip(gcm_years, gcm_months, gcm_days)):
            import datetime
            try:
                ref = datetime.date(2001, m, d)
                gcm_doy[i] = ref.timetuple().tm_yday
            except ValueError:
                gcm_doy[i] = 60  # Feb 29

    da_gcm = da_gcm.assign_coords(common_doy=('time', gcm_doy))

    return da_obs, da_gcm, is_leap_day


# =============================================================================
# STEP 1: DETRENDING (9-year running average)
# =============================================================================

def compute_monthly_running_avg(da_daily, window_years=9):
    """
    For each grid cell × calendar month, compute N-year running average
    of monthly-mean values, mapped back to daily time steps.

    Edge handling: First/last (window//2) years use truncated windows.
    The padding is done with 'edge' mode (repeats boundary values).
    """
    monthly = da_daily.resample(time='1MS').mean()
    vals = monthly.values
    months_arr = monthly.time.dt.month.values

    smoothed = np.full_like(vals, np.nan)
    pad = window_years // 2

    for mon in range(1, 13):
        mask = months_arr == mon
        mon_vals = vals[mask]
        ny = mon_vals.shape[0]

        if ny == 0:
            continue
        if ny < window_years:
            # Not enough years — use the mean of what we have
            smoothed[mask] = np.nanmean(mon_vals, axis=0, keepdims=True)
            continue

        # Pad edges
        padded = np.pad(mon_vals, ((pad, pad), (0, 0), (0, 0)), mode='edge')
        cs = np.nancumsum(padded, axis=0)
        valid_count = np.nancumsum(
            (~np.isnan(padded)).astype(np.float32), axis=0)

        run_sum = cs[window_years:] - cs[:-window_years]
        run_count = valid_count[window_years:] - valid_count[:-window_years]

        with np.errstate(invalid='ignore'):
            run_mean = np.where(run_count > 0, run_sum / run_count, np.nan)

        smoothed[mask] = run_mean[:ny]

    # Map to daily
    daily_years = da_daily.time.dt.year.values
    daily_months = da_daily.time.dt.month.values
    result = np.full_like(da_daily.values, np.nan)

    sm_years = monthly.time.dt.year.values
    sm_months = monthly.time.dt.month.values
    ym_to_idx = {(y, m): i for i, (y, m) in enumerate(zip(sm_years, sm_months))}

    for (y, m), idx in ym_to_idx.items():
        daily_mask = (daily_years == y) & (daily_months == m)
        if daily_mask.any():
            result[daily_mask] = smoothed[idx]

    # Check for any remaining NaN in the running average
    nan_in_result = np.isnan(result) & ~np.isnan(da_daily.values)
    if nan_in_result.any():
        n_problem = nan_in_result.sum()
        print(f"    WARNING: {n_problem} valid data points got NaN running avg")

    return da_daily.copy(data=result)


def step1_detrend(obs_1deg, gcm_his_1deg, window=9, work_dir='.'):
    """
    STEP 1: Remove 9-year running average → anomalies.
    Saves checkpoint after completion.
    """
    ckpt = os.path.join(work_dir, 'step1_detrend.npz')
    if os.path.exists(ckpt):
        print("  STEP 1: Loading from checkpoint...")
        data = np.load(ckpt, allow_pickle=True)
        return dict(data)  # Returns saved arrays

    print("  Computing running averages...")
    t0 = timer.time()

    obs_ra = compute_monthly_running_avg(obs_1deg, window)
    his_ra = compute_monthly_running_avg(gcm_his_1deg, window)

    obs_anom = obs_1deg - obs_ra
    his_anom = gcm_his_1deg - his_ra

    # Climatological running averages per calendar month
    obs_ra_clim = obs_ra.groupby('time.month').mean('time')
    his_ra_clim = his_ra.groupby('time.month').mean('time')

    print(f"    Done in {timer.time()-t0:.1f}s")

    # Check: anomalies should have mean ~0 for each month
    for mon in range(1, 13):
        obs_mon_mean = float(obs_anom.sel(
            time=obs_anom.time.dt.month == mon).mean())
        if abs(obs_mon_mean) > 1.0:
            print(f"    WARNING: Month {mon} obs anomaly mean = "
                  f"{obs_mon_mean:.2f} (expected ~0)")

    # Save checkpoint
    os.makedirs(work_dir, exist_ok=True)
    np.savez_compressed(ckpt,
        obs_anom=obs_anom.values,
        his_anom=his_anom.values,
        obs_ra=obs_ra.values,
        his_ra=his_ra.values,
        obs_ra_clim=obs_ra_clim.values,
        his_ra_clim=his_ra_clim.values,
        obs_time=obs_1deg.time.values,
        his_time=gcm_his_1deg.time.values,
        lat=obs_1deg.lat.values,
        lon=obs_1deg.lon.values,
    )
    print(f"    Checkpoint saved: {ckpt}")

    return {
        'obs_anom': obs_anom,
        'his_anom': his_anom,
        'obs_ra': obs_ra,
        'his_ra': his_ra,
        'obs_ra_clim': obs_ra_clim,
        'his_ra_clim': his_ra_clim,
    }


# =============================================================================
# STEP 2: BIAS CORRECTION — Vectorized QM with calendar-safe DOY
# =============================================================================

def step2_qm(obs_anom, his_anom, obs_ra, land_mask,
             day_window=15, gcm_calendar='noleap', work_dir='.'):
    """
    STEP 2: Quantile mapping with ±15-day window.

    Calendar-safe: uses common_doy (1-365) for both obs and GCM,
    even if obs has leap years and GCM doesn't.

    NaN-safe: explicitly masks NaN values before sorting and quantile
    computation. Never silently drops or fills NaN.

    Vectorized: processes all grid cells per DOY simultaneously.
    """
    ckpt = os.path.join(work_dir, 'step2_qm_his.nc')
    if os.path.exists(ckpt):
        print("  STEP 2: Loading from checkpoint...")
        return xr.open_dataarray(ckpt)

    # Use common_doy coordinate for pooling
    if 'common_doy' in obs_anom.coords:
        obs_doy = obs_anom.common_doy.values
    else:
        obs_doy = obs_anom.time.dt.dayofyear.values

    if 'common_doy' in his_anom.coords:
        his_doy = his_anom.common_doy.values
    else:
        his_doy = his_anom.time.dt.dayofyear.values

    obs_vals = obs_anom.values
    his_vals = his_anom.values

    bc_his_vals = np.full_like(his_vals, np.nan)
    nlat, nlon = obs_vals.shape[1], obs_vals.shape[2]

    n_land = int(land_mask.sum())
    print(f"  Vectorized QM: 365 DOYs × {n_land} land cells")
    t0 = timer.time()

    # Track QM diagnostics
    cells_with_insufficient_data = 0

    for doy in range(1, 366):  # 1-365 (common DOY, no 366)
        if doy % 30 == 0:
            print(f"    DOY {doy}/365  ({timer.time()-t0:.0f}s)")

        # Build ±15 day window (wrapping at year boundaries)
        window_doys = set()
        for d in range(doy - day_window, doy + day_window + 1):
            wrapped = ((d - 1) % 365) + 1  # wrap to 1-365
            window_doys.add(wrapped)
        window_arr = np.array(sorted(window_doys))

        # Pool samples from window
        obs_mask = np.isin(obs_doy, window_arr)
        his_mask = np.isin(his_doy, window_arr)

        obs_pool = obs_vals[obs_mask]  # (n_samples, nlat, nlon)
        his_pool = his_vals[his_mask]

        if obs_pool.shape[0] < 10 or his_pool.shape[0] < 10:
            cells_with_insufficient_data += n_land
            continue

        # Sort pools per cell (axis=0), handling NaN by pushing to end
        # np.sort puts NaN at the end
        obs_sorted = np.sort(obs_pool, axis=0)
        his_sorted = np.sort(his_pool, axis=0)

        # Count valid (non-NaN) samples per cell
        obs_valid_count = (~np.isnan(obs_pool)).sum(axis=0)  # (nlat, nlon)
        his_valid_count = (~np.isnan(his_pool)).sum(axis=0)

        # QM for historical values at this DOY
        his_doy_mask = his_doy == doy
        if not his_doy_mask.any():
            continue

        his_at_doy = his_vals[his_doy_mask]  # (n_years, nlat, nlon)

        for t in range(his_at_doy.shape[0]):
            val_2d = his_at_doy[t]

            # Quantile position: count(his_sorted <= val) / n_valid
            # Only count non-NaN entries in his_sorted
            count_below = np.zeros((nlat, nlon), dtype=np.float32)
            for s in range(his_pool.shape[0]):
                valid_sample = ~np.isnan(his_sorted[s])
                below = valid_sample & (his_sorted[s] <= val_2d)
                count_below += below.astype(np.float32)

            with np.errstate(invalid='ignore'):
                q_pos = np.where(
                    his_valid_count > 0,
                    count_below / his_valid_count.astype(np.float32),
                    np.nan
                )
            q_pos = np.clip(q_pos, 0.0, 1.0)

            # Map to obs distribution
            idx_float = q_pos * (obs_valid_count - 1).astype(np.float32)
            idx_float = np.clip(idx_float, 0, None)
            idx_lo = np.floor(idx_float).astype(int)
            idx_hi = np.minimum(idx_lo + 1, (obs_valid_count - 1).astype(int))
            idx_hi = np.clip(idx_hi, 0, obs_pool.shape[0] - 1)
            idx_lo = np.clip(idx_lo, 0, obs_pool.shape[0] - 1)
            frac = idx_float - idx_lo

            lat_idx, lon_idx = np.meshgrid(
                np.arange(nlat), np.arange(nlon), indexing='ij')
            val_lo = obs_sorted[idx_lo, lat_idx, lon_idx]
            val_hi = obs_sorted[idx_hi, lat_idx, lon_idx]

            bc_val = val_lo * (1 - frac) + val_hi * frac

            # Apply land mask and NaN source mask
            bc_val[~land_mask] = np.nan
            bc_val[np.isnan(val_2d)] = np.nan

            # Find the global time index
            global_idx = np.where(his_doy_mask)[0][t]
            bc_his_vals[global_idx] = bc_val

    elapsed = timer.time() - t0
    print(f"  QM done in {elapsed:.1f}s")

    if cells_with_insufficient_data > 0:
        print(f"    WARNING: {cells_with_insufficient_data} cell-DOY combos "
              f"had insufficient pool data")

    # Restore: add obs running average back
    bc_his = his_anom.copy(data=bc_his_vals) + obs_ra

    # NaN check
    new_nan = np.isnan(bc_his.values) & ~np.isnan(his_anom.values) & land_mask
    if new_nan.any():
        print(f"    WARNING: QM introduced {new_nan.sum()} new NaN values "
              f"on land cells")

    # Save checkpoint
    bc_his.to_netcdf(ckpt)
    print(f"    Checkpoint saved: {ckpt}")

    return bc_his


# =============================================================================
# STEP 3: SPATIAL DISAGGREGATION — IDW scale factors
# =============================================================================

def build_idw_weights(src_lats, src_lons, tgt_lats, tgt_lons,
                      n_neighbors=4, power=2):
    """Build IDW weights using KD-tree. Computed once, reused for all times."""
    src_lon_grid, src_lat_grid = np.meshgrid(src_lons, src_lats)
    src_points = np.column_stack([src_lat_grid.ravel(), src_lon_grid.ravel()])

    tgt_lon_grid, tgt_lat_grid = np.meshgrid(tgt_lons, tgt_lats)
    tgt_points = np.column_stack([tgt_lat_grid.ravel(), tgt_lon_grid.ravel()])

    tree = cKDTree(src_points)
    distances, indices = tree.query(tgt_points, k=n_neighbors)

    with np.errstate(divide='ignore', invalid='ignore'):
        inv_dist = 1.0 / np.power(distances, power)

    exact = distances == 0
    if exact.any():
        inv_dist[exact] = 1e30
        rows_with_exact = exact.any(axis=1)
        for row in np.where(rows_with_exact)[0]:
            inv_dist[row, ~exact[row]] = 0

    row_sums = inv_dist.sum(axis=1, keepdims=True)
    weights = inv_dist / row_sums

    return indices, weights


def step3_spatial_disagg(bc_coarse, obs_fine, obs_coarse, config, work_dir='.'):
    """
    STEP 3: Scale-factor IDW interpolation from 1° to 0.05°.

    Processes year-by-year to manage memory (~2.5GB per year at 0.05°).
    Each year is saved as a separate checkpoint file.
    """
    tgt_lats = obs_fine.lat.values
    tgt_lons = obs_fine.lon.values

    # Daily climatology at both resolutions
    print("  Computing daily climatologies...")
    obs_clim_fine = obs_fine.groupby('time.dayofyear').mean('time')
    obs_clim_coarse = obs_coarse.groupby('time.dayofyear').mean('time')

    # Precompute IDW weights
    idw_idx, idw_wts = build_idw_weights(
        bc_coarse.lat.values, bc_coarse.lon.values,
        tgt_lats, tgt_lons,
        n_neighbors=config.IDW_N_NEIGHBORS,
        power=config.IDW_POWER
    )

    bc_years = bc_coarse.time.dt.year.values
    bc_doy = bc_coarse.time.dt.dayofyear.values
    # Use common_doy if available
    if 'common_doy' in bc_coarse.coords:
        bc_common_doy = bc_coarse.common_doy.values
    else:
        bc_common_doy = bc_doy

    unique_years = np.unique(bc_years)
    output_files = []

    for year in unique_years:
        year_ckpt = os.path.join(work_dir, f'step3_sd_{year}.nc')
        if os.path.exists(year_ckpt):
            print(f"    Year {year}: loading checkpoint")
            output_files.append(year_ckpt)
            continue

        year_mask = bc_years == year
        bc_year = bc_coarse.isel(time=year_mask)
        doys = bc_common_doy[year_mask]

        # Scale factors
        sf_vals = np.full_like(bc_year.values, np.nan)
        for t, doy_val in enumerate(doys):
            sel_doy = doy_val if doy_val in obs_clim_coarse.dayofyear.values else 365
            sf_vals[t] = bc_year.values[t] - obs_clim_coarse.sel(dayofyear=sel_doy).values

        # IDW interpolate scale factors
        ntime_yr = sf_vals.shape[0]
        src_flat = sf_vals.reshape(ntime_yr, -1)
        neighbor_vals = src_flat[:, idw_idx]
        valid = ~np.isnan(neighbor_vals)
        w = np.broadcast_to(idw_wts[np.newaxis], neighbor_vals.shape)
        w_masked = np.where(valid, w, 0)
        w_sum = w_masked.sum(axis=2, keepdims=True)

        with np.errstate(invalid='ignore'):
            sf_fine = np.where(
                w_sum.squeeze(2) > 0,
                (np.nan_to_num(neighbor_vals, 0) * w_masked).sum(axis=2) / w_sum.squeeze(2),
                np.nan
            )
        sf_fine = sf_fine.reshape(ntime_yr, len(tgt_lats), len(tgt_lons))

        # Add fine-resolution climatology
        result_vals = np.full_like(sf_fine, np.nan, dtype=np.float32)
        for t, doy_val in enumerate(doys):
            sel_doy = doy_val if doy_val in obs_clim_fine.dayofyear.values else 365
            result_vals[t] = (sf_fine[t] +
                              obs_clim_fine.sel(dayofyear=sel_doy).values)

        # Save year
        ds_year = xr.Dataset({
            'tmax': xr.DataArray(
                result_vals,
                dims=['time', 'lat', 'lon'],
                coords={
                    'time': bc_year.time.values,
                    'lat': tgt_lats,
                    'lon': tgt_lons,
                }
            )
        })
        ds_year.to_netcdf(year_ckpt)
        output_files.append(year_ckpt)
        print(f"    Year {year}: saved ({ntime_yr} days)")

    return output_files


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def run_bcsd(config=None):
    """
    Run the complete daily BCSD pipeline with all safety checks.
    """
    if config is None:
        config = Config()

    os.makedirs(config.WORK_DIR, exist_ok=True)

    # ── LOAD DATA ─────────────────────────────────────────────────────
    print("=" * 70)
    print("LOADING DATA")
    print("=" * 70)

    print("  Loading CHIRTS 0.05° (for SD step later)...")
    obs_fine = load_yearly_files(
        config.CHIRTS_005_DIR, config.OBS_VAR,
        config.TRAIN_START, config.TRAIN_END)
    check_nan_report(obs_fine, 'CHIRTS_0.05')

    print("  Loading CHIRTS 1° (for QM)...")
    obs_1deg = load_yearly_files(
        config.CHIRTS_1DEG_DIR, config.OBS_VAR,
        config.TRAIN_START, config.TRAIN_END)
    check_nan_report(obs_1deg, 'CHIRTS_1deg')

    print("  Loading GCM 1° (historical)...")
    gcm_his_1deg = load_yearly_files(
        config.GCM_1DEG_DIR, config.GCM_VAR,
        config.TRAIN_START, config.TRAIN_END)
    check_nan_report(gcm_his_1deg, 'GCM_1deg')

    # ── VERIFY GRIDS ──────────────────────────────────────────────────
    print("\n  Grid alignment checks:")
    verify_grid_alignment(obs_1deg, gcm_his_1deg, 'CHIRTS_1deg', 'GCM_1deg')

    # ── CALENDAR ALIGNMENT ────────────────────────────────────────────
    print("\n  Calendar alignment:")
    obs_1deg, gcm_his_1deg, leap_mask = align_calendars(
        obs_1deg, gcm_his_1deg, config.GCM_CALENDAR)

    # ── SHAPEFILE MASKING ─────────────────────────────────────────────
    land_mask = None
    if config.SHAPEFILE_PATH and os.path.isdir(config.SHAPEFILE_PATH):
        print("\n  Applying shapefile mask...")
        obs_1deg, mask_2d = apply_shapefile_mask(obs_1deg, config.SHAPEFILE_PATH)
        gcm_his_1deg = gcm_his_1deg.where(~np.isnan(mask_2d))
        land_mask = ~np.isnan(mask_2d).values
    else:
        # Use obs data coverage as land mask
        land_mask = ~np.all(np.isnan(obs_1deg.values), axis=0)
    print(f"    Land cells: {land_mask.sum()}")

    # ── STEP 1: DETREND ───────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 1: DETRENDING")
    print("=" * 70)
    dt = step1_detrend(obs_1deg, gcm_his_1deg,
                       config.RUNNING_AVG_WINDOW, config.WORK_DIR)

    # ── STEP 2: BIAS CORRECTION ───────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 2: BIAS CORRECTION (QM)")
    print("=" * 70)
    bc_his = step2_qm(
        dt['obs_anom'], dt['his_anom'], dt['obs_ra'],
        land_mask, config.QM_DAY_WINDOW, config.GCM_CALENDAR,
        config.WORK_DIR
    )

    # ── STEP 3: SPATIAL DISAGGREGATION ────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 3: SPATIAL DISAGGREGATION (IDW)")
    print("=" * 70)
    output_files = step3_spatial_disagg(
        bc_his, obs_fine, obs_1deg, config, config.WORK_DIR)

    print("\n" + "=" * 70)
    print(f"PIPELINE COMPLETE — {len(output_files)} yearly files saved")
    print(f"Output dir: {config.WORK_DIR}")
    print("=" * 70)

    return output_files


# =============================================================================
# USAGE
# =============================================================================
if __name__ == '__main__':
    print("""
    Edit Config class paths, then call:
        from bcsd_daily_v3 import run_bcsd, Config
        cfg = Config()
        cfg.CHIRTS_005_DIR = '/kaggle/input/...'
        output_files = run_bcsd(cfg)
    """)